# GPU feasibility spike: robust DFL dispatch layer, k=1 (original (T,T) formulation)

**Status as of this version**: `solver=cp.CUCLARABEL` is confirmed backed by **Clarabel.jl
(Julia)**, specifically the `CuClarabel` branch, bridged into Python via `juliacall` -- NOT a
pure-Python/JAX implementation, though the differentiation/backward pass separately involves
JAX (`lineax`/`equinox`) and a package called `diffqcp`. Confirmed working setup sequence
(hands-on, not guessed):
1. `uv pip install --system juliacall==0.9.31 cupy-cuda13x diffqcp cvxpy cvxpylayers torch`
2. Install the `CuClarabel` branch of `Clarabel.jl` + Julia's `CUDA` package via `juliacall`
3. `os.environ["JULIA_CUDA_USE_BINARYBUILDER"] = "true"` -- forces Julia's CUDA.jl to use its
   own BinaryBuilder-provided CUDA binaries instead of picking up Python's pip-installed
   `nvidia-*` libraries. This directly addresses an earlier CUDA.jl warning about runtime
   libraries being loaded from a Python system path -- a real Julia/Python CUDA-runtime
   conflict risk when both language runtimes try to manage CUDA state in the same process.
4. `jax.config.update("jax_enable_x64", True)` -- JAX defaults to float32 unless told
   otherwise; `GAMMA=1e-6` (this problem's Tikhonov coefficient) is close to float32's
   precision floor, so this rules out precision as a confound before anything else runs.

**Prior results, WITHOUT this full setup** (all on the SAME original, unmodified `(T,T)`
formulation -- the lower-triangular reformulation explored earlier was already ruled out as
the cause, since these same failures reproduced on the original formulation too):
- B=16: OOM (~8.52GiB requested for one operation)
- B=1 (well below any memory pressure): NaN/singular-operator error from `lineax`'s linear
  solve

This notebook now performs the full confirmed setup FIRST, then retests cleanly.

**Runtime**: Set Runtime -> Change runtime type -> T4 GPU (or A100 if available) before
running. Run cells top-to-bottom -- setup must complete before anything calls
`solver=cp.CUCLARABEL`.

In [ ]:
!nvidia-smi

In [ ]:
# --- Part A dependencies (high confidence -- well-established packages) ---
!uv pip install --system juliacall==0.9.31 cupy-cuda13x diffqcp cvxpy cvxpylayers torch

In [ ]:
# --- Julia/CuClarabel setup (REQUIRED before any solver=cp.CUCLARABEL call) ---
# Confirmed by hands-on testing: CUCLARABEL is backed by Clarabel.jl (Julia), the
# "CuClarabel" branch specifically, bridged via juliacall -- NOT a pure-Python/JAX
# implementation. JULIA_CUDA_USE_BINARYBUILDER=true forces Julia's CUDA.jl to use its own
# BinaryBuilder-provided CUDA binaries instead of picking up Python's pip-installed
# nvidia-* libraries -- this directly addresses the earlier CUDA.jl warning about runtime
# libraries being loaded from a Python system path (a real Julia/Python CUDA-runtime
# conflict risk, not just cosmetic).
import os
from juliacall import Main as jl
jl.seval('using Pkg; Pkg.add(url="https://github.com/oxfordcontrol/Clarabel.jl", rev="CuClarabel")')
jl.seval('using Pkg; Pkg.add("CUDA")')
os.environ["JULIA_CUDA_USE_BINARYBUILDER"] = "true"

In [ ]:
import os
# JAX pre-allocates ~75-90% of free GPU memory by default the first time it touches the GPU
# (XLA_PYTHON_CLIENT_PREALLOCATE=true is the default). With Julia's CUDA.jl ALSO claiming
# its own memory pool in the same process, the two runtimes likely each grab large chunks of
# the same GPU upfront, leaving very little headroom -- plausible explanation for OOM at
# B=2 despite the actual tensors needing only ~1GB. Must be set BEFORE jax touches the GPU.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Optional additional lever if OOM persists even with the above: cap JAX's pool explicitly,
# e.g. os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.3" (leaves ~70% for Julia/CUDA.jl).

import jax

# Force JAX to use 64-bit precision globally -- part of BASE setup, not an isolated A/B
# test. GAMMA=1e-6 is close to float32's precision floor (~1.19e-7); this rules out
# precision as a confound before any of the actual diagnostic tests further down run.
jax.config.update("jax_enable_x64", True)

In [ ]:
import cvxpy as cp
import torch
print("cvxpy:", cp.__version__, " has CUCLARABEL symbol:", hasattr(cp, "CUCLARABEL"))
print("installed_solvers (CPU-visible ones):", cp.installed_solvers())
print("torch:", torch.__version__, " cuda available:", torch.cuda.is_available())

import cvxpylayers
print("cvxpylayers:", getattr(cvxpylayers, "__version__", "unknown"))

import numpy as np
import time
from cvxpylayers.torch import CvxpyLayer

## Root cause diagnosis, and why this version drops `mu_p`/`mu_m` entirely

Traced the actual call chain in the installed `cvxpylayers`/`diffqcp` source (no GPU needed --
it's plain Python, downloaded and inspected locally):

- `cvxpylayers/interfaces/cuclarabel_if.py`'s `_compute_gradients()` calls
  `qcp_module_instance.vjp(dx, dy, ds)` with **no `solve_method` argument** -- there is a
  `# TODO(quill): add ability to pass parameters to vjp` sitting right next to that call.
- `diffqcp`'s `DeviceQCP.vjp` (the concrete GPU class actually used here) defaults
  `solve_method` to **`"jax-lu"`**, not `"jax-lsmr"`.
- Inside `_vjp_common`, `"jax-lsmr"` routes through `lineax`'s `LSMR` -- an iterative
  least-squares solver built to handle singular / rank-deficient operators gracefully.
  `"jax-lu"` takes the OTHER branch: it densifies the KKT operator and calls plain
  `lineax.linear_solve()` with no solver specified, which **assumes the operator is
  genuinely nonsingular**. That is exactly the code path whose failure mode is
  `"A linear solver returned non-finite (NaN or inf) output"`.

The underlying operator genuinely IS singular in the original box-dual formulation,
independent of the exact data: `mu_p`/`mu_m` (the box constraint's LP duals) are nonneg
variables that never appear in the objective, so at the true optimum a large fraction of
them sit *exactly* at zero wherever the corresponding `A_ij = 0`. That's a large
flat/degenerate subspace in the KKT system, sitting at a non-smooth nonneg-cone boundary.

**We are NOT patching `diffqcp`/`cvxpylayers` to force `"jax-lsmr"`** -- ruled out as a
project constraint. Regularizing `mu_p`/`mu_m` (`gamma_mu * sum_squares(...)`) was tried as a
problem-level fix instead, and tested extensively on CPU: it made backward ~1.4-1.9x
*slower*, not faster (`diffcp`'s LSQR already handles the degenerate block gracefully via a
minimum-norm solution, so removing it bought nothing and the denser Hessian cost more than it
saved).

**This version replaces the box/`mu_p`/`mu_m` formulation entirely** with an ellipsoidal
uncertainty set, weighted by the model's actual `Sigma_xi_chol` (not a generic unit ball --
`{xi : ||Sigma_xi_chol^-1 xi|| <= omega}`), robustified via Cauchy-Schwarz into a single SOC
constraint per direction with **no auxiliary dual variables at all**:
`A0 + omega * ||A @ Sigma_xi_chol||_2 <= B`. This removes the degenerate block at its root --
there is no `mu_p`/`mu_m`-like block sitting at a nonneg-cone boundary with zero objective
curvature, because there's no LP duality involved at all. (This construction is also, via a
different theoretical route, the tight conservative approximation of the analogous CVaR
chance constraint, at `omega = sqrt((1-eps)/eps)` -- see Georghiou, Kuhn & Wiesemann (2019),
Section 2.2, and Ben-Tal/Nemirovski-style ellipsoidal robust optimization for the two
equivalent derivations.)

**CPU numbers already measured for this exact formulation** (batch=1, ECOS, local): forward
9.6s, backward 15.5s, total 25.2s -- comparable to (slightly better than) the original
`mu_p`/`mu_m` formulation (~23-29s), and confirmed DPP-compliant. Since it lacks the specific
degeneracy diagnosed above, this GPU test is a genuine correctness test of whether that
degeneracy was the actual GPU blocker.

## Problem construction

Ellipsoidal (`Sigma_xi_chol`-weighted) robust reformulation of the k=1 corner -- NOT what's
currently in `6_models/models_robust/dispatch_layer_robust.py` (that file still has the
original box/`mu_p`/`mu_m` formulation; this notebook is where the ellipsoidal alternative is
being tested first). No `h_plus`/`h_minus`, no auxiliary dual variables at all -- just
`pl_hat` and `Sigma_xi_chol` as params, matching the non-robust `dispatch_layer.py`'s param
list exactly. 1224 total problem variables (vs. 8136 for the box/`mu_p`/`mu_m` version) --
`D_ch`/`D_dis` remain plain `(T,T)` Variables with the usual `upper_tri(.) == 0` equality.
SYNTHETIC data (right shapes/scales, not real forecasts) -- this is a
conditioning/correctness test, not an accuracy test.

In [ ]:
T, N = 24, 64
ETA_CH, ETA_DIS, C_CH, C_DIS, B_MAX, SOC0, DT = 0.95, 0.95, 2.0, 2.0, 4.0, 2.0, 1.0
GAMMA = 1e-6   # matches dispatch_layer_robust.py's default, applied to primal variables.
OMEGA = 2.0    # ellipsoid radius / robustness multiplier. See OMEGA sweep further down --
               # unlike GAMMA_MU in the earlier mu_p/mu_m version, this isn't chasing a
               # singularity fix (there's no degenerate block here at all); it's a
               # calibration sensitivity check.

def ellipsoidal_constraint(A0, A, B, Sigma_xi_chol, omega):
    """Robustify A0 + A@xi <= B against xi in the Sigma_xi_chol-weighted ellipsoid
    {xi : ||Sigma_xi_chol^-1 xi|| <= omega}, via Cauchy-Schwarz. NO auxiliary dual
    variables -- replaces the mu_p/mu_m box-duality construction entirely. Also the tight
    conservative CVaR bound for the analogous chance constraint at omega=sqrt((1-eps)/eps)
    (Georghiou, Kuhn & Wiesemann 2019, Sec 2.2)."""
    row_norms = cp.norm(A @ Sigma_xi_chol, p=2, axis=1)   # (T,)
    return [A0 + omega * row_norms <= B]

def build_ellipsoidal_k1_problem(omega=OMEGA):
    p_ch_hat  = cp.Variable(T, name="p_ch_hat")
    p_dis_hat = cp.Variable(T, name="p_dis_hat")
    D_ch      = cp.Variable((T, T), name="D_ch")
    D_dis     = cp.Variable((T, T), name="D_dis")
    p_da_rel  = cp.Variable(T, name="p_da_rel")

    pl_hat  = cp.Parameter(T, name="pl_hat")
    Sigma_xi_chol = cp.Parameter((T, T), name="Sigma_xi_chol")

    cons = [cp.upper_tri(D_ch) == 0, cp.upper_tri(D_dis) == 0]

    power_flow_hat = ETA_CH * p_ch_hat - (1.0 / ETA_DIS) * p_dis_hat
    s_hat = SOC0 + DT * cp.cumsum(power_flow_hat)
    D_net = ETA_CH * D_ch - (1.0 / ETA_DIS) * D_dis
    G = DT * cp.cumsum(D_net, axis=0)

    cons += ellipsoidal_constraint(p_ch_hat,  D_ch,  C_CH, Sigma_xi_chol, omega)
    cons += ellipsoidal_constraint(-p_ch_hat, -D_ch, 0.0,  Sigma_xi_chol, omega)
    cons += ellipsoidal_constraint(p_dis_hat,  D_dis,  C_DIS, Sigma_xi_chol, omega)
    cons += ellipsoidal_constraint(-p_dis_hat, -D_dis, 0.0,   Sigma_xi_chol, omega)
    cons += ellipsoidal_constraint(s_hat,  G,  B_MAX, Sigma_xi_chol, omega)
    cons += ellipsoidal_constraint(-s_hat, -G, 0.0,   Sigma_xi_chol, omega)

    cons += [s_hat[T - 1] == SOC0, G[T - 1, :] == 0]
    cons += [p_da_rel + pl_hat >= -5.0, p_da_rel + pl_hat <= 11.0]

    imb_det = p_ch_hat - p_dis_hat - p_da_rel
    R = np.eye(T) + D_ch - D_dis

    sum_trace = DT**2 * (cp.sum_squares(imb_det) + cp.sum_squares(R @ Sigma_xi_chol))
    penalty = GAMMA * (cp.sum_squares(p_ch_hat) + cp.sum_squares(p_dis_hat)
                        + cp.sum_squares(D_ch) + cp.sum_squares(D_dis) + cp.sum_squares(p_da_rel))

    prob = cp.Problem(cp.Minimize(sum_trace + penalty), cons)
    assert prob.is_dcp(dpp=True), "not DPP"

    params = [pl_hat, Sigma_xi_chol]
    variables = [p_ch_hat, p_dis_hat, D_ch, D_dis, p_da_rel]
    return prob, params, variables

prob, params, variables = build_ellipsoidal_k1_problem()
print(f"problem vars: {sum(v.size for v in prob.variables())}  (expect 1224 -- no mu_p/mu_m, no auxiliary variables at all)")

In [ ]:
def synthetic_batch(B, device="cpu", seed=0):
    """Right shapes/scales. No h_plus/h_minus needed -- the ellipsoidal formulation uses
    Sigma_xi_chol directly (the model's actual second-moment estimate) rather than a
    separate quantile-derived box."""
    torch.manual_seed(seed)
    pl_hat = (torch.randn(B, T, dtype=torch.float64, device=device) * 1.5).requires_grad_(True)
    xi = torch.randn(B, N, T, dtype=torch.float64, device=device) * 0.6
    xi = (xi - xi.mean(dim=1, keepdim=True)).requires_grad_(True)
    M = xi.transpose(-2, -1) @ xi / N
    M = 0.5 * (M + M.transpose(-2, -1)) + 1e-9 * torch.eye(T, dtype=torch.float64, device=device)
    Sigma_xi_chol = torch.linalg.cholesky(M)
    return pl_hat, Sigma_xi_chol

def time_forward_backward(layer, args, label, solver_args=None):
    t0 = time.time()
    dec = layer(*args, solver_args=solver_args) if solver_args else layer(*args)
    if torch.cuda.is_available() and args[0].is_cuda:
        torch.cuda.synchronize()   # GPU ops are async -- must sync before stopping the timer
    t_fwd = time.time() - t0
    loss = sum((d**2).sum() for d in dec)
    t1 = time.time()
    loss.backward()
    if torch.cuda.is_available() and args[0].is_cuda:
        torch.cuda.synchronize()
    t_bwd = time.time() - t1
    print(f"  {label:30s} forward={t_fwd:8.3f}s  backward={t_bwd:8.3f}s  total={t_fwd+t_bwd:8.3f}s")
    return t_fwd, t_bwd

## Part A: CPU baseline (classic `solver_args={"solve_method": ...}` path)

Already measured locally for this exact ellipsoidal formulation (batch=1, ECOS): forward
9.6s, backward 15.5s, total 25.2s. For reference, the original box/`mu_p`/`mu_m` formulation
measured (WSL2, real data, batch=16): ECOS 483.9s (37.0s fwd + 446.9s bwd), Clarabel 535.0s,
SCS 1443.8s. This Part A run (batch=16, Colab CPU) is a sanity check that Colab's CPU is in
the same ballpark as WSL2's/local's, not a new speed benchmark for this formulation --
we already have that number.

In [ ]:
B = 16
layer_cpu = CvxpyLayer(prob, parameters=params, variables=variables)
print(f"Part A -- original (T,T) formulation, Colab CPU, batch={B}")
for solver in ["ECOS", "Clarabel"]:
    args = synthetic_batch(B, device="cpu", seed=1)
    try:
        time_forward_backward(layer_cpu, args, solver, solver_args={"solve_method": solver})
    except Exception as e:
        print(f"  {solver}: FAILED -- {type(e).__name__}: {e}")

## Part B: GPU-native via `cvxpylayers`' `solver=cp.CUCLARABEL` -- ellipsoidal formulation test

This is the actual test: does the ellipsoidal (`Sigma_xi_chol`-weighted, no auxiliary dual
variables) formulation succeed on `diffqcp`'s default `"jax-lu"` dense solve, where the
`mu_p`/`mu_m` box-dual formulation failed with `"A linear solver returned non-finite (NaN or
inf) output"` at every batch size tried previously (B=1 through B=16, before and after the
full Julia/CUDA/precision setup, with and without `mu_p`/`mu_m` regularization). If this
formulation succeeds where the box-dual one failed, that confirms the degenerate
zero-curvature block was the actual GPU blocker -- not batching, not precision, not solver
setup. No `diffqcp`/`cvxpylayers` internals are touched -- this is a change to the
optimization problem itself, run through the standard `solver=cp.CUCLARABEL` path.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}, OMEGA: {OMEGA}")

results = {}
for B_try in [1, 2, 4, 8, 16]:
    print(f"\n--- B={B_try} ---")
    try:
        layer_gpu = CvxpyLayer(prob, parameters=params, variables=variables, solver=cp.CUCLARABEL).to(device)
        args_gpu = synthetic_batch(B_try, device=device, seed=1)
        t_fwd, t_bwd = time_forward_backward(layer_gpu, args_gpu, f"CUCLARABEL B={B_try}")
        results[B_try] = ("OK", t_fwd, t_bwd)
    except Exception as e:
        msg = str(e)
        if "non-finite" in msg or "singular" in msg.lower():
            kind = "NaN/singular (conditioning)"
        elif "RESOURCE_EXHAUSTED" in msg or "out of memory" in msg.lower() or "OOM" in msg:
            kind = "OOM (memory)"
        else:
            kind = f"OTHER: {type(e).__name__}"
        print(f"  FAILED -- {kind}")
        results[B_try] = (kind, None, None)
    if device == "cuda":
        torch.cuda.empty_cache()   # clear fragmented allocations before the next attempt

print("\n" + "=" * 60)
print(f"SUMMARY (OMEGA={OMEGA})")
for B_try, (status, t_fwd, t_bwd) in results.items():
    if status == "OK":
        print(f"  B={B_try:3d}: OK  forward={t_fwd:.3f}s backward={t_bwd:.3f}s")
    else:
        print(f"  B={B_try:3d}: {status}")

print("\nInterpretation:")
print("- If ANY B succeeds: the ellipsoidal formulation avoids the singularity -- confirms")
print("  the mu_p/mu_m degenerate block was the actual GPU blocker, not something else")
print("  (batching/precision/setup already ruled out earlier). Largest successful B = the")
print("  real memory ceiling. Compare total time to Part A's CPU number (25.2s) for a")
print("  genuine speed verdict.")
print("- If EVERY B fails with NaN/singular: the degeneracy hypothesis was wrong, or there's")
print("  a DIFFERENT singularity source unrelated to mu_p/mu_m -- run the OMEGA sweep next,")
print("  but if that also fails, this points to something more fundamental (e.g. the CUDA")
print("  version mismatch lead from earlier -- PyTorch cu128 vs system/CuPy cu13x).")
print("- If EVERY B fails with OOM (even B=1): memory-viability issue, independent of the")
print("  singularity question -- see the earlier XLA_PYTHON_CLIENT_PREALLOCATE fix.")

## If Part B still fails at every B: sweep OMEGA, then reconsider other leads

Unlike the earlier `mu_p`/`mu_m` version, this formulation has no known degenerate block, so
there's less reason to expect OMEGA's magnitude specifically to matter for a NaN/singular
failure -- this sweep is mainly a sensitivity check (does the result hold across reasonable
calibration choices), not a "make it more nonsingular" search. If every OMEGA value still
fails the same way, that's a real signal the earlier CUDA-version-mismatch lead (PyTorch
cu128 vs system/CuPy cu13x, from earlier this session) or something else entirely is the
actual blocker, not anything specific to the box/`mu_p`/`mu_m` formulation.

In [ ]:
# --- OMEGA sweep: sensitivity check, and a fallback if the default fails ---
for omega_try in [1.0, 2.0, 4.36]:   # 4.36 = sqrt((1-eps)/eps) at eps=0.05, the rigorous
                                       # worst-case-moment CVaR calibration (see Sec 2.2 markdown)
    print(f"\n--- OMEGA={omega_try} ---")
    prob_o, params_o, variables_o = build_ellipsoidal_k1_problem(omega=omega_try)
    try:
        layer_gpu_o = CvxpyLayer(prob_o, parameters=params_o, variables=variables_o, solver=cp.CUCLARABEL).to(device)
        args_gpu_o = synthetic_batch(1, device=device, seed=1)   # B=1, the smallest previously-failing case
        time_forward_backward(layer_gpu_o, args_gpu_o, f"OMEGA={omega_try}")
        print(f"  >>> SUCCESS at OMEGA={omega_try}")
    except Exception as e:
        print(f"  FAILED -- {type(e).__name__}: {e}")

print("\nIf NONE of these succeed: the ellipsoidal reformulation does not fix the GPU")
print("failure either -- the mu_p/mu_m degeneracy hypothesis was likely NOT the (sole)")
print("cause. Worth revisiting the CUDA-version-mismatch lead (PyTorch cu128 vs system/CuPy")
print("cu13x) from earlier this session before concluding the GPU path is unusable outright.")
print("If SOME succeed: note which OMEGA -- check downstream (in the real training pipeline,")
print("not this notebook) whether that calibration is a reasonable robustness/coverage level")
print("before adopting it for real training.")

## Verdict checklist

- [ ] Setup cells ran clean: `uv pip install` succeeded, Julia/CuClarabel installed,
      `JULIA_CUDA_USE_BINARYBUILDER=true` set, `jax_enable_x64` confirmed `True`.
- [ ] Part A total (Colab CPU, ECOS, batch=16) -- broadly comparable to the WSL2 baseline
      (483.9s) and to the local batch=1 number (25.2s, scaled)?
- [ ] Part B batch-size ramp (ellipsoidal formulation, OMEGA=2.0): any B succeed, or still
      NaN/singular at every B?
- [ ] If it still fails at every B: does the OMEGA sweep (1.0, 2.0, 4.36) succeed at any
      value?
- [ ] If Part B succeeded (any B, any OMEGA): real GPU total vs. Part A's CPU total (25.2s)
      -- genuine win? What's the largest B that fits without OOM?
- [ ] If a win is confirmed: does it hold up at B=16 (the real training batch size)?

If NOTHING in the OMEGA sweep resolves the singularity, that's strong evidence the
`mu_p`/`mu_m` degeneracy was never the (sole) cause of the GPU failure -- worth revisiting
the CUDA-version-mismatch lead (PyTorch cu128 vs system/CuPy cu13x) before concluding the GPU
path is unusable for this problem entirely. Either way, note that even a full GPU win here
(any formulation, any batch size) only helps the *robust* corner specifically -- CPU numbers
already show the non-robust formulation is ~15x faster than this ellipsoidal one even before
any GPU speedup, so the GPU question and the "is the robust model worth training at all"
question are separate and both still open.